In [ ]:
import requests, pandas as pd
#quote is the url encoder
from urllib.parse import quote
from eea import disco

In [ ]:
#TBL so I don't have to rewrite every time
TBL = "[CO2Emission].[latest].[co2cars_2025Pv31]"

So now we'll do some EDA to figure out which columns are necessary. Which specs will I want to predict WLTP CO2 from?
I want to train RF/XGBoost to predict WLTP gCO₂/km from vehicle attributes on EEA data
First I'll just get a quick overview of the data by looking at some inital rows.

In [ ]:
disco(f"SELECT TOP 100 * "
f"FROM {TBL}")

,ID,MS,Mp,VFN,Mh,Man,MMS,TAN,T,Va,...,Year,Status,Version_file,E (g/km),Er (g/km),Zr,Dr,Fc,Ech,RLFI
0,175878466,DE,HYUNDAI MOTOR EUROPE,IP-050662-NLH,HYUNDAI TURKIYE,HYUNDAI MOTOR TURKIYE OTOMOTIV AS,None,E5*2007/46*0121*06,BC3,B5P51,...,2025,P,v31,None,None,None,2025-12-22,5.2,EA,RL-050009-NLH
1,175901181,DE,HYUNDAI MOTOR EUROPE,IP-050662-NLH,HYUNDAI TURKIYE,HYUNDAI MOTOR TURKIYE OTOMOTIV AS,None,E5*2007/46*0121*06,BC3,B5P51,...,2025,P,v31,None,None,None,2025-05-26,5.2,EA,RL-050009-NLH
2,176038548,DE,HYUNDAI MOTOR EUROPE,IP-050662-NLH,HYUNDAI TURKIYE,HYUNDAI MOTOR TURKIYE OTOMOTIV AS,None,E5*2007/46*0121*06,BC3,B5P51,...,2025,P,v31,None,None,None,2025-12-19,5.2,EA,RL-050009-NLH
3,176039670,DE,HYUNDAI MOTOR EUROPE,IP-050662-NLH,HYUNDAI TURKIYE,HYUNDAI MOTOR TURKIYE OTOMOTIV AS,None,E5*2007/46*0121*06,BC3,B5P51,...,2025,P,v31,None,None,None,2025-05-28,5.2,EA,RL-050009-NLH
4,176052774,DE,HYUNDAI MOTOR EUROPE,IP-050662-NLH,HYUNDAI TURKIYE,HYUNDAI MOTOR TURKIYE OTOMOTIV AS,None,E5*2007/46*0121*06,BC3,B5P51,...,2025,P,v31,None,None,None,2025-08-25,5.2,EA,RL-050009-NLH
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,176512774,DE,HYUNDAI MOTOR EUROPE,IP-050662-NLH,HYUNDAI TURKIYE,HYUNDAI MOTOR TURKIYE OTOMOTIV AS,None,E5*2007/46*0121*06,BC3,B5P51,...,2025,P,v31,None,None,None,2025-10-01,5.2,EA,RL-050009-NLH
96,176514993,DE,HYUNDAI MOTOR EUROPE,IP-050662-NLH,HYUNDAI TURKIYE,HYUNDAI MOTOR TURKIYE OTOMOTIV AS,None,E5*2007/46*0121*06,BC3,B5P51,...,2025,P,v31,None,None,None,2025-12-09,5.2,EA,RL-050009-NLH
97,176540477,DE,HYUNDAI MOTOR EUROPE,IP-050662-NLH,HYUNDAI TURKIYE,HYUNDAI MOTOR TURKIYE OTOMOTIV AS,None,E5*2007/46*0121*06,BC3,B5P51,...,2025,P,v31,None,None,None,2025-02-03,5.2,EA,RL-050009-NLH
98,176545927,DE,HYUNDAI MOTOR EUROPE,IP-050662-NLH,HYUNDAI TURKIYE,HYUNDAI MOTOR TURKIYE OTOMOTIV AS,None,E5*2007/46*0121*06,BC3,B5P51,...,2025,P,v31,None,None,None,2025-06-20,5.2,EA,RL-050009-NLH


In [ ]:
#How many rows are there?
disco(f"SELECT COUNT(*) AS n_rows " 
f"FROM {TBL}")

,n_rows
0,10833597


In [ ]:
#target distribution (name the aggregate, or it errors)
Ewltp = "[Ewltp (g/km)]"
disco(f"SELECT Ft AS fuel_type, COUNT(*) AS n, AVG(CAST({Ewltp} AS float)) AS mean_co2 " 
f"FROM {TBL}" 
f"GROUP BY Ft " 
f"ORDER BY n DESC")

,fuel_type,n,mean_co2
0,petrol,6075402,128.027738
1,electric,2055863,0.000000
2,diesel,1279640,151.832064
3,petrol/electric,1009882,31.796999
4,lpg,352268,119.812034
5,diesel/electric,47590,36.344846
6,e85,12384,128.497253
7,hydrogen,451,0.000000
8,ng,108,107.240741
9,unknown,9,184.285714


In [ ]:
#cardinality of the scary columns
disco(f"SELECT COUNT(DISTINCT Mk) AS n_Mk_makes, COUNT(DISTINCT Cn) AS n_Cn_names " 
f"FROM {TBL}")

,n_Mk_makes,n_Cn_names
0,500,7273


Now we'll get into inspecting some impotant columns more closely to see if there's anything wrong with them, or to see if they need to be adressed before we can train the model.
This includes missing data or potential data leakage that can impact the accuracy of our model.

In [ ]:
#missingness, one column at a time, starting with Specific CO2 Emissions (WLTP).
disco(f"SELECT COUNT(*) AS n_null " 
f"FROM {TBL}" 
f"WHERE {Ewltp} IS NULL")

,n_null
0,6526


So Ewltp (g/km) has about 0.06% missing values. Since this is what we're predicting it doesn't matter too much but it's good that there's not that many missing values.

In [ ]:
#Electric energy consumption.
Z = "[Z (Wh/km)]"
disco(f"SELECT {Z}"
f"FROM {TBL}")

,Z (Wh/km)
0,None
1,None
2,None
3,None
4,None
...,...
995,None
996,None
997,None
998,None


In [ ]:
disco(f"SELECT COUNT(*) AS n_null " 
f"FROM {TBL}" 
f"WHERE {Z} IS NULL")

,n_null
0,7885389


So electric energy consumption (Z (Wh/km)) doesn't appear to have any useful information. The vast majority of its rows are empty, so we won't be using it to train our model.

In [ ]:
#Mass in running order Completed/complete vehicle .
M = "[M (kg)]"
disco(f"SELECT {M}"
f"FROM {TBL}")

,M (kg)
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN
...,...
995,915.0
996,915.0
997,915.0
998,915.0


In [ ]:
#Mass in running order Completed/complete vehicle .
disco(f"SELECT "
f"COUNT(*) - COUNT({M})                          AS n_null, "
f"100.0 * (COUNT(*) - COUNT({M})) / COUNT(*)     AS pct_null, "
f"COUNT(*)                                     AS n_total "
f"FROM {TBL}")

,n_null,pct_null,n_total
0,329,0.003037,10833597


This is great. Mass essentially has no missing rows, so we can just impute the missing rows before we train the model.

In [ ]:
#Engine capacity.
Ec = "[Ec (cm3)]"
disco(f"SELECT "
f"COUNT(*) - COUNT({Ec})                          AS n_null, "
f"100.0 * (COUNT(*) - COUNT({Ec})) / COUNT(*)     AS pct_null, "
f"COUNT(*)                                     AS n_total "
f"FROM {TBL}")

,n_null,pct_null,n_total
0,2056378,18.981489,10833597


Ok so for engine capacity, there's a lot more nulls here.

In [ ]:
#Engine power.
Ep = "[Ep (KW)]"
disco(f"SELECT "
f"COUNT(*) - COUNT({Ep})                          AS n_null, "
f"100.0 * (COUNT(*) - COUNT({Ep})) / COUNT(*)     AS pct_null, "
f"COUNT(*)                                     AS n_total "
f"FROM {TBL}")

,n_null,pct_null,n_total
0,459236,4.238998,10833597


In [ ]:
#Category of the vehicle registered.
disco(f"SELECT "
f"COUNT(*) - COUNT(Cr)                          AS n_null, "
f"100.0 * (COUNT(*) - COUNT(Cr)) / COUNT(*)     AS pct_null, "
f"COUNT(*)                                     AS n_total "
f"FROM {TBL}")

,n_null,pct_null,n_total
0,0,0.0,10833597


In [ ]:
#Fuel type
disco(f"SELECT "
f"COUNT(*) - COUNT(Ft)                          AS n_null, "
f"100.0 * (COUNT(*) - COUNT(Ft)) / COUNT(*)     AS pct_null, "
f"COUNT(*)                                     AS n_total "
f"FROM {TBL}")

,n_null,pct_null,n_total
0,0,0.0,10833597


So we examined the most important columns. Thus, to predict WLTP CO2 from engineering specs we'll eventually want to use the following columns:
- M (kg), Mt
- Ec (cm3), Ep (KW)
- Ft, Fm
- W (mm), At1, At2
- Ct / Cr
- Zr
- Mh or Mk (ordinal encode rather than one hot)
- Year, MS

Exclude (leakage):
- Fc
- Z (Wh/km)
- Enedc, Ernedc, Er
- De, Vf

But for a first simple model, we'll just use:
- Ft in {petrol, diesel}
- M (kg)
- Ec (cm3)
- Ep (KW)
- Cr

In [ ]:
disco(f"SELECT COUNT(*) AS c FROM (SELECT DISTINCT {Ewltp}, Ft, Cr, {M}, {Ec}, {Ep} FROM {TBL}) AS d")

,c
0,118342


There's 10.8 million rows in the dataset in total, but these are registration events. We just care about distinct spec combinations. When we count the distinct spec combination and target we care about, the table reduces to only 188k rows, which is far more manageable. Accounting for distinct spec combinations also prevents identical row leakage. So we can now create our dataframe.

In [ ]:
disco(f"SELECT {Ewltp}, Ft, Cr, {M}, {Ec}, {Ep}, COUNT(*) AS n_registrations "
f"FROM {TBL} "
f"GROUP BY {Ewltp}, Ft, Cr, {M}, {Ec}, {Ep}")

,Ewltp (g/km),Ft,Cr,M (kg),Ec (cm3),Ep (KW),n_registrations
0,None,diesel,M1,NaN,NaN,12.0,1
1,None,diesel,M1,NaN,1968.0,90.0,1
2,None,diesel,M1,NaN,1968.0,142.0,1
3,None,diesel,M1,NaN,1995.0,110.0,1
4,None,diesel,M1,NaN,1997.0,81.0,1
...,...,...,...,...,...,...,...
995,None,petrol/electric,M1,2495.0,2998.0,230.0,2
996,None,petrol/electric,M1,2500.0,2995.0,224.0,1
997,None,petrol/electric,M1,2515.0,2995.0,260.0,3
998,None,petrol/electric,M1,2530.0,2995.0,224.0,2


So now we can create the actual table that we'll train the model on.
(this part maybe not using: "we'll use DuckDB becuase embedded + OLAP (column store, analytical)")

In [ ]:
df = disco(f"SELECT DISTINCT {Ewltp}, Ft, Cr, {M}, {Ec}, {Ep} FROM {TBL}", n=120000)
print(df.shape)

(118342, 6)


In [ ]:
df.head()

,Ewltp (g/km),Ft,Cr,M (kg),Ec (cm3),Ep (KW)
0,NaN,diesel,M1,NaN,NaN,12.0
1,NaN,diesel,M1,NaN,1968.0,90.0
2,NaN,diesel,M1,NaN,1968.0,142.0
3,NaN,diesel,M1,NaN,1995.0,110.0
4,NaN,diesel,M1,NaN,1997.0,81.0


In [ ]:
df.dtypes          # JSON nulls turn numeric columns into `object` — very likely here
df.isna().sum()
df.describe()